In [ ]:
input_data = None
input_doc = None
output_data = None
util = None
display_util = None

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import matplotlib_inline

from IPython.display import Markdown

matplotlib_inline.backend_inline.set_matplotlib_formats("svg")


%matplotlib inline
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", 500)
pd.set_option("display.max_columns", None)

sys.path.append(str(Path(util).parent))
sys.path.append(str(Path(display_util).parent))

In [ ]:
from display_util import rename, display_data_doc, collist  # noqa: E402
from util import (  # noqa: E402
    drop_col_few_distinct,
    drop_duplicate_columns,
)

# transplantation

This file provides information on the direct preperation of the operation, the operation itself and the discharge from the transplantation centre of the patient. Most columns are organ specific. Data is provided by {term}`ET` and {term}`IQTIG` in the file `element_transplantation.csv`. We perform the common steps outlined in [](../02_preprocessing/index.md).

## Unprocessed input data

In [ ]:
data = pd.read_csv(input_data, sep=";", low_memory=False)
display_data_doc(data=data, official_doc=pd.read_csv(input_doc))

## Technical Steps

For this file the general plan for technical preprocessing was followed.

### Removal of Non-Informative Columns

First we tried to remove empty and duplicated columns (see [](general:ecf)). 

In [ ]:
data = drop_col_few_distinct(data)
data = drop_duplicate_columns(data)
dropme = [
    "TTransplantationszentrumET",
    "TInstitutionskennzeichenIQTIG",
    "TTransplantationszentrumRegistrierungET",
    "TBetriebsstaettennummerIQTIG",
    "TEntlassungStandortIQTIG",
    "TFachabteilungIQTIG",
    "TFollowUpZentrumET",
]
display(
    Markdown(
        f"""Furthemore we removed the columns {collist(dropme)}, because they contained encoded identifying information.
"""
    )
)
data = data.drop(columns=dropme)
del dropme

### Renaming of Columns

New names were used for the columns. (see [](general:cr)).

In [ ]:
renaming = {
    "TAbbruchTxIQTIG": "aborted",
    "TAufnahmeKrankenhausDateIQTIG": "hospitilization_date",
    "TAufnahmeWartelisteDateET": "admission_date",
    "TAzathioprinHGabeIQTIG": "heart_azathioprin_1",
    "TAzathioprinLuGabeIQTIG": "lung_azathioprin_1",
    "TBestimmungsortET": "destination",
    #    "TBetriebsstaettennummerIQTIG": "location_iqtig",
    "TCreatinkinaseHWertIQTIG": "creatine_kinase_h_u_per_l",
    "TCreatinkinaseMBHWertIQTIG": "creatine_kinase_mbh_u_per_l",
    "TCyclosporinHGabeIQTIG": "cyclosporin_heart_1",
    "TCyclosporinLuGabeIQTIG": "cyclosporin_lung_1",
    "TDrainagePgangET": "pancreatic_drainage",
    "TDrainageVenenET": "vene_drainage",
    "TDringlET": "waiting_state",
    "TDringlHIQTIG": "waiting_state_heart_iqtig",
    "TDringlLeIQTIG": "waiting_state_liver_iqtig",
    "TDringlLuIQTIG": "waiting_state_lung_iqtig",
    "TEinzelOderDoppelTransplantationNIQTIG": "kidney_single_or_both",
    "TEntlassungAzathioprinHGabeIQTIG": "heart_azathioprin_2",
    "TEntlassungAzathioprinLuGabeIQTIG": "lung_azathioprin_2",
    "TEntlassungCyclosporinHGabeIQTIG": "cyclosporin_heart_2",
    "TEntlassungCyclosporinLuGabeIQTIG": "cyclosporin_lung_2",
    "TEntlassungDiagnoseELTRLeIQTIG": "liver_eltr",
    "TEntlassungDiagnoseICD10BeschreibungIQTIG": "icd10",
    "TEntlassungFEV1LuWertIQTIG": "lung_fev1_percent",
    "TEntlassungFunktionTransplantatNIQTIG": "kidney_function",
    "TEntlassungGrundIQTIG": "discharge_reason_code",
    "TEntlassungHunterstuetzungssystemKunstherzHIQTIG": "artificial_heart_support",
    "TEntlassungImmunsuppressivaAndereHGabeIQTIG": "heart_other_immunsupressiva_2",
    "TEntlassungImmunsuppressivaAndereLuGabeIQTIG": "lung_other_immunsupressiva_2",
    "TEntlassungInsulinfreiPIQTIG": "pancreas_insulin_free",
    "TEntlassungKrankenhausDateIQTIG": "discharge_date",
    "TEntlassungMToRHemmerHGabeIQTIG": "heart_mtor_suppr_release",
    "TEntlassungMToRHemmerLuGabeIQTIG": "lung_mtor_suppr_release",
    "TEntlassungMycophenolatHGabeIQTIG": "heart_mycophenolat_release",
    "TEntlassungMycophenolatLuGabeIQTIG": "lung_mycophenolat_release",
    #    "TEntlassungStandortIQTIG": "discharge_location_iqtig",
    "TEntlassungSteroideHGabeIQTIG": "heart_steroids_1",
    "TEntlassungSteroideLuGabeIQTIG": "lung_sterioids_1",
    "TEntlassungTacrolimusHGabeIQTIG": "heart_tacrolimus_1",
    "TEntlassungTacrolimusLuGabeIQTIG": "lung_tacrolimus_1",
    "TEntlassungTracheotomieLuIQTIG": "lung_tracheotomie",
    "TEntnahmeTransplantatPIQTIG": "pancreas_removal_necessary",
    "TEntnahmeTransplantatUrsachePIQTIG": "pancreas_removal_reason",
    # "TFachabteilungIQTIG": "departement",
    "TFollowUpLetztesDateET": "last_followup_date",
    "TFollowUpLostToFollowUpET": "lost_to_followup",
    #   "TFollowUpZentrumET": "follow_up_location",
    "THaematokritHWertIQTIG": "heart_haematokrit_percent",
    "THypotensivePeriodeHIQTIG": "heart_hypotension_period",
    "TIdEmpfaengerNrETET": "recipient_et_id_et",
    "TIdEmpfaengerNrETIQTIG": "recipient_et_iqtig",
    "TIdSpenderNrETET": "donor_et_id_et",
    "TIdSpenderNrETIQTIG": "donor_et_iqtig",
    "TIdTransplantationsnummerETET": "transplant_et_id",
    "TImmunsuppression1ET": "immunosuppression_a",
    "TImmunsuppression2ET": "immunosuppression_b",
    "TImmunsuppression3ET": "immunosuppression_c",
    "TImmunsuppression4ET": "immunosuppression_d",
    "TImmunsuppressionInitial1ET": "immunosuppression_a_initial",
    "TImmunsuppressionInitial2ET": "immunosuppression_b_initial",
    "TImmunsuppressionInitial3ET": "immunosuppression_c_initial",
    "TImmunsuppressionInitial4ET": "immunosuppression_d_initial",
    "TImmunsuppressivaAndereHGabeIQTIG": "heart_other_immunsupressiva_1",
    "TImmunsuppressivaAndereLuGabeIQTIG": "lung_other_immunsupressiva_1",
    "TImplantationsstelleET": "implant_side",
    "TInduktionstherapieHIQTIG": "heart_induction_therapie",
    "TInduktionstherapieLuIQTIG": "lung_induction_therapie",
    "TIschaemiezeitGesamtLuWertIQTIG": "total_ischemia_time_lung_min",
    "TIschaemiezeitKaltHWertIQTIG": "cold_ischemia_time_heart_min",
    "TIschaemiezeitKaltLeWertIQTIG": "cold_ischemia_time_liver_min",
    "TIschaemiezeitKaltWertET": "cold_ischemia_time_min",
    "TIschaemiezeitWarmZweiteWertET": "warm_ischemia_time_min",
    "TKatecholamintherapieHIQTIG": "heart_catecholamine_therapy",
    "TKomplikationIntraPostOperationAllgmeinNPIQTIG": "kidney_pancreas_complications_general",
    "TMToRInhibitorHGabeIQTIG": "heart_mtor_suppr",
    "TMToRInhibitorLuGabeIQTIG": "lung_mtor_suppr",
    "TMycophenolatHGabeIQTIG": "heart_mycophenolat",
    "TMycophenolatLuGabeIQTIG": "lung_mycophenolat",
    "TOCSSystemHIQTIG": "heart_ocs",
    "TOperationSimultanLuIQTIG": "lung_simul_operation",
    "TOPSCodeBeschreibungIQTIG": "ops_code",
    "TOrganET": "organ",
    "TOrganfunktionInitialET": "initial_organ_function",
    "TOrganqualitaetHIQTIG": "organ_quality",
    "TOrganteilLeIQTIG": "liver_parts",
    "TPostOPAbstossungHBehandeltAnzahlIQTIG": "heart_rejections_treated",
    "TPostOPAbstossungNIQTIG": "kidney_rejection",
    "TPostOPAbstossungPIQTIG": "pancreas_rejection",
    "TPostOPAnzahlDialysenNIQTIG": "dialysis_count",
    "TPostOPFunktionsaufnahmeTransplantatNIQTIG": "post_operation_functional",
    "TPostOPHarnausscheidungErsteStundeWertET": "first_hour_urine_production_ml_per_hour",
    "TPostOPHarnausscheidungMengeRestWertET": "remaining_urine_production_ml_per_hour",
    "TPostOPHarnausscheidungMengeWertET": "urine_production_ml_per_hour",
    "TPostOPKreatininNWertIQTIG": "creatinine_post_operation_umol_per_l",
    "TPostOPOrganversagenDateET": "postop_organ_failure_date",
    "TPostOPOrganversagenExplantationDateET": "date_explanation_of_failure",
    "TPostOPOrganversagenUrsacheET": "failure_reason",
    "TPostOPTodesursacheHIQTIG": "heart_cause_of_death",
    "TPostOPTodesursacheLeIQTIG": "liver_cause_of_death",
    "TPostOPTodesursacheLuIQTIG": "lung_cause_of_death",
    "TPostOPTodesursacheNIQTIG": "kidney_cause_of_death",
    "TRelaparotomieNPIQTIG": "kidney_pancreas_relaparotomy",
    "TRelaparotomieUrsacheNPIQTIG": "kidney_pancreas_relaparotomy_cause",
    "TRetransplantationLuIQTIG": "lung_retransplant",
    "TRetransplantationNIQTIG": "kidney_retransplant",
    "TRetransplantationPIQTIG": "pancreas_retransplant",
    "TSpendeKompatibelNPIQTIG": "kidney_pancreas_donor_compatible",
    "TSteroideHGabeIQTIG": "heart_steroids_2",
    "TSteroideLuGabeIQTIG": "lung_sterioids_2",
    "TStillstandHIQTIG": "cardiac_arrest",
    "TTacrolimusHGabeIQTIG": "heart_tacrolimus_2",
    "TTacrolimusLuGabeIQTIG": "lung_tacrolimus_2",
    "TTransplantationArtLuIQTIG": "lung_operation_type",
    "TTransplantationArtPNIQTIG": "kidney_pancreas_operation_type",
    "TTxDateET": "operation_date_et",
    "TTxDateIQTIG": "operation_date_iqtig",
    "TVergabeProgramET": "assignment_program",
    "TVergabeTypET": "assignment_type",
    "TZentrumsangebotLeIQTIG": "liver_center_offer",
    "TAcceptableMismatchProgrammET": "acceptable_mismatch_program",
    "TBlutungenNPIQTIG": "kidney_pancreas_bleeding",
    "TCreatinkinaseHEinheitIQTIG": "creatine_kinase_h_unit",
    "TCreatinkinaseMBHEinheitIQTIG": "creatine_kinase_mbh_unit",
    "TEntlassungFEV1LuEinheitIQTIG": "lung_fev1_unit",
    "THaematokritHEinheitIQTIG": "heart_haematokrit_unit",
    "TIschaemiezeitGesamtLuEinheitIQTIG": "total_ischemia_time_lung_unit",
    "TIschaemiezeitKaltEinheitET": "cold_ischemia_time_unit",
    "TIschaemiezeitKaltHEinheitIQTIG": "cold_ischemia_time_heart_unit",
    "TIschaemiezeitKaltLeEinheitIQTIG": "cold_ischemia_time_liver_unit",
    "TIschaemiezeitWarmZweiteEinheitET": "warm_ischemia_time_unit",
    "TKomplikationSonstigeNPIQTIG": "kidney_pancreas_complications_other",
    "TPostOPAbstossungHBehandeltUnbekanntAnzahlIQTIG": "heart_rejections_treated_unknown_number",
    "TPostOPHarnausscheidungErsteStundeEinheitET": "first_hour_urine_production_unit",
    "TPostOPHarnausscheidungMengeEinheitET": "urine_production_unit",
    "TPostOPHarnausscheidungMengeRestEinheitET": "remaining_urine_production_unit",
    "TPostOPKreatininNEinheitIQTIG": "creatinine_post_operation_unit",
    "TPostOPTodesursacheNLebIQTIG": "kidney_liver_deathreason_postop",
    "TReoperationNPIQTIG": "reoperation_kidney_pancreas",
    "TTransplantationPartiellET": "transplantation_partial",
    "TTransplantationTechnikET": "transplantation_protocol",
    "TTransplantationTechnikPET": "transplantation_pancreas_protocol",
}
data = rename(data, renaming)
data.to_parquet(output_data)